# Quantifying Forecast Accuracy: The Brier Score

In quantitative research, we rarely deal with certainties. Instead, we deal with **probabilities**. But how do you measure if a probabilistic forecast is actually "good"?

The **Brier Score** is the gold standard for this. Unlike simple accuracy (which only asks if you were right or wrong), the Brier Score asks: *How confident were you in the right answer?*

## The Mathematics of Certainty

The Brier Score is essentially the Mean Squared Error (MSE) of a probability forecast:

$$\text{BS} = \frac{1}{N} \sum_{t=1}^{N} (f_t - o_t)^2$$

- $f_t$: The forecast probability (e.g., $0.7$ for a 70% chance of a price increase).
- $o_t$: The actual outcome ($1$ if the price increased, $0$ if it didn't).

### Interpreting the Result
- **0.0**: Perfect forecast (you assigned 100% probability to everything that happened).
- **0.25**: The "Naive" score (assigning 50% to everything—essentially a coin flip).
- **1.0**: Worst possible forecast (you were 100% sure the opposite would happen).

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd

# Get 1 year of BTC data for a meaningful sample
btc = yf.download('BTC-USD', period='1y', interval='1d')
df = btc[['Close']].copy()

# Outcome (o_t): 1 if price went UP tomorrow, 0 otherwise
df['outcome'] = (df['Close'].shift(-1) > df['Close']).astype(int)
df = df.dropna()

## Scenario 1: The Naive Baseline

If we have no predictive model, we might assume a constant 50% probability of an increase every day. This is our baseline for comparison.

In [ ]:
df['naive_prob'] = 0.5
naive_bs = np.mean((df['naive_prob'] - df['outcome'])**2)
print(f"Naive Brier Score: {naive_bs:.4f}")

## Scenario 2: A Simple Momentum Rule

Let's test a primitive strategy: **Momentum**. 
We assume that the market has short-term inertia:
- If the price went up yesterday, we forecast a **60%** chance it goes up today.
- If it went down yesterday, we forecast a **40%** chance it goes up today.

In [ ]:
# Determine if yesterday was positive (relative to the day before)
df['yesterday_up'] = (df['Close'].shift(1) > df['Close'].shift(2)).astype(int)

# Assign probabilities based on momentum rule
df['momentum_prob'] = np.where(df['yesterday_up'] == 1, 0.6, 0.4)

# Clean up shifted NaNs
df_clean = df.dropna()
momentum_bs = np.mean((df_clean['momentum_prob'] - df_clean['outcome'])**2)
print(f"Momentum Brier Score: {momentum_bs:.4f}")

## Final Comparison

Comparing the two strategies reveals whether our simple momentum rule actually adds value:

| Strategy | Brier Score | Interpretation |
|---|---|---|
| Naive | 0.2500 | No predictive power |
| Momentum | *(Calculated above)* | Relative to naive |

**The Takeaway:**
If the Momentum score is **lower** than 0.25, the strategy is better than a coin flip. If it is **higher**, the strategy is actively misleading (it is "confident" about the wrong things).

In quantitative research, don't just track accuracy. Use the Brier Score to ensure your model's confidence matches the reality of the market.